In [ ]:
# S0 — Imports
import os
import time
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

_WALL_START = time.time()

In [ ]:
# S1 — Paths
# Change these paths to your real mounted input paths

DATA_DIR = Path("/kaggle/input/notebooks/tiantanghuaxiao/bridclef-2026-offline")

PATH_EMB = DATA_DIR / "emb_full_probe.npy"
PATH_TARGETS = DATA_DIR / "student_targets_window.npy"
PATH_MASK = DATA_DIR / "student_mask_window.npy"
PATH_SOFT = DATA_DIR / "student_teacher_probs_window.npy"
PATH_META = DATA_DIR / "student_meta_window.parquet"

print("emb exists:", PATH_EMB.exists())
print("targets exists:", PATH_TARGETS.exists())
print("mask exists:", PATH_MASK.exists())
print("soft exists:", PATH_SOFT.exists())
print("meta exists:", PATH_META.exists())

In [ ]:
# S2 — Load data
emb_full_probe = np.load(PATH_EMB).astype(np.float32)
student_targets_window = np.load(PATH_TARGETS).astype(np.int8)
student_mask_window = np.load(PATH_MASK).astype(np.uint8)
student_teacher_probs_window = np.load(PATH_SOFT).astype(np.float32)
student_meta_window = pd.read_parquet(PATH_META)

N_ROWS, EMB_DIM = emb_full_probe.shape
N_CLASSES = student_teacher_probs_window.shape[1]

print("emb_full_probe:", emb_full_probe.shape, emb_full_probe.dtype)
print("student_targets_window:", student_targets_window.shape, student_targets_window.dtype)
print("student_mask_window:", student_mask_window.shape, student_mask_window.dtype)
print("student_teacher_probs_window:", student_teacher_probs_window.shape, student_teacher_probs_window.dtype)
print("student_meta_window:", student_meta_window.shape)

In [ ]:
# S3 — Basic diagnostics
print("masked supervised entries:", int(student_mask_window.sum()))
print("positive pseudo labels:", int((student_targets_window == 1).sum()))
print("negative pseudo labels:", int((student_targets_window == 0).sum()))
print("ignored entries:", int((student_targets_window == -1).sum()))

per_class_supervised = student_mask_window.sum(axis=0)
print("classes with any supervised entries:", int((per_class_supervised > 0).sum()))
print("max supervised count in one class:", int(per_class_supervised.max()))
print("min positive supervised count among active classes:", int(per_class_supervised[per_class_supervised > 0].min()))

In [ ]:
# S4 — Fit scaler + PCA for student inputs
scaler_student = StandardScaler()
X_scaled = scaler_student.fit_transform(emb_full_probe)

PCA_DIM = 128
pca_student = PCA(n_components=PCA_DIM, random_state=42)
X_pca = pca_student.fit_transform(X_scaled)

print("X_pca:", X_pca.shape)

In [ ]:
# S5 — Train per-class student regressors on teacher soft probabilities
# We only train on rows where student_mask_window[:, j] == 1

student_active_class_idx = np.where(student_mask_window.sum(axis=0) > 0)[0].astype(np.int32)

student_coef_mat = np.zeros((len(student_active_class_idx), PCA_DIM), dtype=np.float32)
student_intercept_vec = np.zeros(len(student_active_class_idx), dtype=np.float32)

for k, j in enumerate(tqdm(student_active_class_idx, desc="Training student regressors")):
    mask_j = student_mask_window[:, j].astype(bool)

    Xj = X_pca[mask_j]
    yj = student_teacher_probs_window[mask_j, j].astype(np.float32)

    # lightweight soft-target regression
    reg = Ridge(alpha=1.0, random_state=42)
    reg.fit(Xj, yj)

    student_coef_mat[k] = reg.coef_.astype(np.float32)
    student_intercept_vec[k] = np.float32(reg.intercept_)

print("student_active_class_idx:", student_active_class_idx.shape)
print("student_coef_mat:", student_coef_mat.shape)
print("student_intercept_vec:", student_intercept_vec.shape)

In [ ]:
# S6 — Offline fit quality check (not leaderboard, just sanity)
student_pred = np.zeros((len(X_pca), N_CLASSES), dtype=np.float32)

student_pred[:, student_active_class_idx] = (
    X_pca @ student_coef_mat.T + student_intercept_vec[None, :]
).astype(np.float32)

# clip to probability range since ridge can go outside [0,1]
student_pred = np.clip(student_pred, 0.0, 1.0)

mse_all = ((student_pred - student_teacher_probs_window) ** 2)[student_mask_window.astype(bool)].mean()
mae_all = np.abs(student_pred - student_teacher_probs_window)[student_mask_window.astype(bool)].mean()

print("masked MSE:", float(mse_all))
print("masked MAE:", float(mae_all))

In [ ]:
# S7 — Save student artifact
student_artifact = {
    "pca_dim": PCA_DIM,

    "scaler_mean": scaler_student.mean_.astype(np.float32),
    "scaler_scale": scaler_student.scale_.astype(np.float32),

    "pca_mean": pca_student.mean_.astype(np.float32),
    "pca_components": pca_student.components_.astype(np.float32),

    "student_active_class_idx": student_active_class_idx.astype(np.int32),
    "student_coef_mat": student_coef_mat.astype(np.float32),
    "student_intercept_vec": student_intercept_vec.astype(np.float32),
}

with open("student_artifact_v1.pkl", "wb") as f:
    pickle.dump(student_artifact, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Saved student_artifact_v1.pkl")
print("File exists:", Path("student_artifact_v1.pkl").exists())
print("File size (MB):", Path("student_artifact_v1.pkl").stat().st_size / 1024 / 1024)

In [ ]:
# S8 — Quick verification
with open("student_artifact_v1.pkl", "rb") as f:
    chk = pickle.load(f)

for k in [
    "scaler_mean", "scaler_scale",
    "pca_mean", "pca_components",
    "student_active_class_idx", "student_coef_mat", "student_intercept_vec"
]:
    v = chk[k]
    print(k, v.shape, getattr(v, "dtype", type(v)))

In [ ]:
# S9 — Final timing
wall_time = time.time() - _WALL_START
print("Wall time min:", round(wall_time / 60, 2))